# CelebA CLAQ Experiment

A notebook for the canonical CelebA CLAQ workflow.

Main choices in this path:

- prediction target: `Attractive`
- query vocabulary: CelebA attributes excluding `Attractive`
- sensitive set: a small gender-presentation subset
- Concept-QA supervision: direct per-image CelebA attributes
- analysis: replay examples first, then one minimal fixed-history comparison

If CelebA is not already present under `data/`, set `download_celeba = True` in the setup cell once.


In [2]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch

from claq.analysis import (
    plot_lambda_tradeoff_summary,
    plot_rollout_comparisons,
    sample_intuition_replays,
)
from claq.config import CelebAClaqConfig, default_paths
from claq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_run_bundle,
    make_sensitive_mask,
    save_bundle_checkpoint,
)
from claq.data import (
    get_celeba_concept_qa_loaders,
    get_celeba_datasets,
    get_celeba_loaders,
    get_raw_celeba_dataset,
    load_celeba_attribute_spec,
)
from claq.models import ConceptNet2
from claq.sensitive_labels import (
    build_sensitive_labels_from_concept_targets,
    load_sensitive_labels,
    save_sensitive_labels,
)
from claq.training import HistorySamplingConfig, build_claq_models, fit_concept_qa, fit_claq, seed_everything


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = CelebAClaqConfig()
device = config.device
seed_everything(config.random_seed)

figure_ext = ".svg"
download_celeba = False
experiment_name = "celeba_attractive_gender_rs16_anneal_lam04"
qa_experiment_name = "celeba_attractive"

train_min_history = 0
train_max_history = 16
train_non_sensitive_only = False
sample_sensitive_attribute = "Male"
sensitive_target_mode = "max"
actor_eps_end = 0.2
actor_eps_anneal_epochs = config.default_train_epochs
claq_lambda_s = 0.4

concept_qa_max_train_batches = 80 if device.type == "cpu" else None
concept_qa_max_eval_batches = 20 if device.type == "cpu" else None
claq_max_train_batches = 80 if device.type == "cpu" else None
claq_max_eval_batches = 20 if device.type == "cpu" else None


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


{
    "device": str(device),
    "experiment_name": experiment_name,
    "qa_experiment_name": qa_experiment_name,
}


{'device': 'cpu',
 'experiment_name': 'celeba_attractive_gender_rs16_anneal_lam04',
 'qa_experiment_name': 'celeba_attractive'}

In [5]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
spec = load_celeba_attribute_spec(
    root=paths.data_root,
    target_attribute=config.target_attribute,
    sensitive_attributes=config.sensitive_attributes,
    download=download_celeba,
)
positive_class_idx = 1
positive_class_name = spec.class_names[positive_class_idx]
concepts = spec.concept_names
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sens_idx = spec.sensitive_indices
sensitive_mask = make_sensitive_mask(len(concepts), sens_idx, device)
sample_sens_idx = torch.tensor([spec.query_attribute_names.index(sample_sensitive_attribute)], dtype=torch.long)

qa_train_loader, qa_valid_loader = get_celeba_concept_qa_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    download=download_celeba,
)
claq_train_loader, claq_valid_loader, _ = get_celeba_loaders(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    return_query_targets=True,
    download=download_celeba,
)
_, _, test_ds = get_celeba_datasets(
    transform=preprocess,
    root=paths.data_root,
    spec=spec,
    return_query_targets=False,
    download=download_celeba,
)
raw_test_ds = get_raw_celeba_dataset(paths.data_root, spec=spec, split="test", download=False)

print(f"target attribute: {spec.target_attribute}")
print(f"# queries: {len(concepts)}")
print(f"sensitive query attributes: {spec.sensitive_attribute_names}")
print(f"sample sensitive target: {sample_sensitive_attribute}")
print(f"positive class for plots: {positive_class_name}")
print(f"train/valid/test sizes: {len(claq_train_loader.dataset)}, {len(claq_valid_loader.dataset)}, {len(test_ds)}")


target attribute: Attractive
# queries: 39
sensitive query attributes: ['Male', 'No_Beard', 'Mustache', 'Goatee', 'Sideburns', '5_o_Clock_Shadow', 'Heavy_Makeup', 'Wearing_Lipstick']
sample sensitive target: Male
positive class for plots: attractive
train/valid/test sizes: 162770, 19867, 19962


In [6]:
qa_checkpoint = paths.checkpoints_root / f"concept_qa_{qa_experiment_name}.pt"
qa_history_path = paths.runs_root / f"concept_qa_{qa_experiment_name}_history.json"

if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
    qa_source = qa_checkpoint
else:
    qa_model = ConceptNet2().to(device)
    qa_optimizer = torch.optim.Adam(qa_model.parameters(), lr=config.learning_rate)
    qa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        qa_optimizer,
        T_max=max(config.concept_qa_epochs, 1),
    )
    qa_history = fit_concept_qa(
        model=qa_model,
        train_loader=qa_train_loader,
        eval_loader=qa_valid_loader,
        optimizer=qa_optimizer,
        scheduler=qa_scheduler,
        num_epochs=config.concept_qa_epochs,
        model_clip=model_clip,
        dictionary=dictionary,
        class_concept_targets=None,
        clip_device=device,
        train_device=device,
        max_train_batches=concept_qa_max_train_batches,
        max_eval_batches=concept_qa_max_eval_batches,
    )
    torch.save(qa_model.state_dict(), qa_checkpoint)
    with open(qa_history_path, "w", encoding="utf-8") as handle:
        json.dump(qa_history, handle, indent=2)
    answering_model = qa_model.eval()
    qa_source = qa_checkpoint

print(f"Concept-QA ready from: {qa_source}")


Concept-QA ready from: /Users/amir.atashin/Projects/claq/artifacts/models/concept_qa_celeba_attractive.pt


In [7]:
sensitive_labels_dir = paths.artifacts_root / "sensitive_labels" / experiment_name

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels_from_concept_targets(
        concept_targets=claq_train_loader.dataset.query_targets,
        sens_idx=sens_idx,
    )
    s_soft_test, s_hard_test = build_sensitive_labels_from_concept_targets(
        concept_targets=test_ds.query_targets,
        sens_idx=sens_idx,
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Query-sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Mean query-sensitive activation (train/test):",
    float(sensitive_label_cache["s_soft_train"].mean()),
    float(sensitive_label_cache["s_soft_test"].mean()),
)
print(f"Lambda sample-sensitive target: {sample_sensitive_attribute}")


Query-sensitive labels ready from: cache -> /Users/amir.atashin/Projects/claq/artifacts/sensitive_labels/celeba_attractive_gender_rs16_anneal_lam04
{'s_soft_train': (162770,), 's_hard_train': (162770,), 's_soft_test': (19962,), 's_hard_test': (19962,)}
Mean query-sensitive activation (train/test): 0.29746267199516296 0.299725741147995
Lambda sample-sensitive target: Male


In [8]:
def load_or_train_bundle(
    run_name,
    lambda_s,
    lambda_c,
    min_history=train_min_history,
    max_history=train_max_history,
    non_sensitive_only=train_non_sensitive_only,
    epochs=config.default_train_epochs,
    learning_rate=config.learning_rate,
    max_train_batches=claq_max_train_batches,
    max_eval_batches=claq_max_eval_batches,
    force_retrain=False,
):
    ckpt_path = paths.checkpoints_root / f"{experiment_name}_{run_name}_best.pt"
    history_path = paths.runs_root / f"{experiment_name}_{run_name}_history.json"
    if ckpt_path.exists() and not force_retrain:
        return load_run_bundle(
            ckpt_path,
            device=device,
            max_queries=len(concepts),
            num_classes=config.num_classes,
            actor_eps=config.actor_eps,
        )

    actor, classifier, s_head = build_claq_models(
        max_queries=len(concepts),
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=claq_train_loader,
        test_loader=claq_valid_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=lambda_s,
        lambda_c=lambda_c,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_eval_batches,
        actor_eps_end=actor_eps_end,
        actor_eps_anneal_epochs=actor_eps_anneal_epochs,
        sensitive_target_mode=sensitive_target_mode,
        sensitive_target_indices=sample_sens_idx,
    )
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        metadata={
            "run_name": run_name,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "target_attribute": spec.target_attribute,
            "sample_sensitive_attribute": sample_sensitive_attribute,
            "query_sensitive_attributes": spec.sensitive_attribute_names,
            "sensitive_target_mode": sensitive_target_mode,
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
            "actor_eps_start": config.actor_eps,
            "actor_eps_end": actor_eps_end,
            "actor_eps_anneal_epochs": actor_eps_anneal_epochs,
            "actor_state_dict": best["actor_state_dict"],
            "classifier_state_dict": best["classifier_state_dict"],
            "s_head_state_dict": best["s_head_state_dict"],
        },
    )
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    return load_run_bundle(
        ckpt_path,
        device=device,
        max_queries=len(concepts),
        num_classes=config.num_classes,
        actor_eps=config.actor_eps,
    )


baseline_bundle = load_or_train_bundle("baseline", lambda_s=0.0, lambda_c=0.0)
claq_bundle = load_or_train_bundle("lam_0.40", lambda_s=claq_lambda_s, lambda_c=0.0)

print(baseline_bundle["ckpt_path"])
print(claq_bundle["ckpt_path"])


def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )


/Users/amir.atashin/Projects/claq/artifacts/models/celeba_attractive_gender_rs16_anneal_lam04_baseline_best.pt
/Users/amir.atashin/Projects/claq/artifacts/models/celeba_attractive_gender_rs16_anneal_lam04_lam_0.40_best.pt


In [13]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    claq_bundle=claq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=spec.class_names,
    num_cases=6,
    pool_size=500 if device.type == "cpu" else 1500,
    prefer_baseline_sensitive=False,
    confidence_threshold=config.confidence_threshold,
    rollout_max_steps=8,
    positive_class_idx=positive_class_idx,
    positive_class_name=positive_class_name,
    balance_labels=True,
    balance_concept_idx=int(sample_sens_idx.item()),
    balance_concept_name=sample_sensitive_attribute,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path(f"{experiment_name}_rollout_replay_examples"),
    title_prefix="rollout replay",
)

print(f"Saved rollout replay figure: {intuition_fig}")


Sampling intuition replays: 100%|██████████| 500/500 [00:23<00:00, 21.11it/s]


Saved rollout replay figure: /Users/amir.atashin/Projects/claq/artifacts/figures/celeba_attractive_gender_rs16_anneal_lam04_rollout_replay_examples.svg


## Lambda Trade-Off Sweep

Train or load a small lambda sweep and summarize the accuracy vs sensitive-query-rate trade-off from the saved CLAQ histories.


In [16]:
lambda_sweep_values = [0.0, 0.2, 0.4]


def lambda_run_name(lambda_s):
    return "baseline" if lambda_s == 0.0 else f"lam_{lambda_s:.2f}"


sweep_bundles = {
    lambda_run_name(lambda_s): load_or_train_bundle(
        lambda_run_name(lambda_s),
        lambda_s=lambda_s,
        lambda_c=0.0,
    )
    for lambda_s in lambda_sweep_values
}


def load_sweep_history(run_name):
    history_path = paths.runs_root / f"{experiment_name}_{run_name}_history.json"
    with open(history_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def summarize_sweep_run(lambda_s):
    run_name = lambda_run_name(lambda_s)
    history = load_sweep_history(run_name)
    # Report the converged operating point (final epoch, after eps annealing) so runs are
    # comparable; picking the max-val-acc epoch cherry-picks early high-exploration epochs.
    final_row = max(history, key=lambda row: row["epoch"])
    return {
        "run_name": run_name,
        "lambda_s": lambda_s,
        "epoch": final_row["epoch"],
        "test_acc": final_row["test_acc"],
        "test_sens_q_rate": final_row["test_sens_q_rate"],
        "test_loss": final_row["test_loss"],
        "test_q_entropy": final_row["test_q_entropy"],
        "actor_eps": final_row["actor_eps"],
    }


lambda_sweep_rows = [summarize_sweep_run(lambda_s) for lambda_s in lambda_sweep_values]
lambda_sweep_path = paths.runs_root / f"{experiment_name}_lambda_tradeoff_summary.json"
with open(lambda_sweep_path, "w", encoding="utf-8") as handle:
    json.dump(lambda_sweep_rows, handle, indent=2)

lambda_sweep_rows


[{'run_name': 'baseline',
  'lambda_s': 0.0,
  'epoch': 5,
  'test_acc': 0.7131423969396486,
  'test_sens_q_rate': 0.4663322359323502,
  'test_loss': 0.7897690981626511,
  'test_q_entropy': 6.999858692324779e-06,
  'actor_eps': 0.19999999999999996},
 {'run_name': 'lam_0.20',
  'lambda_s': 0.2,
  'epoch': 5,
  'test_acc': 0.7042331504504958,
  'test_sens_q_rate': 0.13183831349015235,
  'test_loss': 0.868851175904274,
  'test_q_entropy': 6.999858692324779e-06,
  'actor_eps': 0.19999999999999996},
 {'run_name': 'lam_0.40',
  'lambda_s': 0.4,
  'epoch': 5,
  'test_acc': 0.6939145316353752,
  'test_sens_q_rate': 0.0,
  'test_loss': 0.8795503556728363,
  'test_q_entropy': 6.999858692324779e-06,
  'actor_eps': 0.19999999999999996}]

In [17]:
lambda_tradeoff_figure = plot_lambda_tradeoff_summary(
    lambda_sweep_rows,
    output_path=figure_path(f"{experiment_name}_lambda_tradeoff"),
)

print(f"Saved lambda sweep summary: {lambda_sweep_path}")
print(f"Saved lambda trade-off figure: {lambda_tradeoff_figure}")


Saved lambda sweep summary: /Users/amir.atashin/Projects/claq/artifacts/runs/celeba_attractive_gender_rs16_anneal_lam04_lambda_tradeoff_summary.json
Saved lambda trade-off figure: /Users/amir.atashin/Projects/claq/artifacts/figures/celeba_attractive_gender_rs16_anneal_lam04_lambda_tradeoff.svg
